# Matrix Transformation Animation
Visualisasi transformasi matriks — titik-titik bergerak mendekati sumbu X.

**Transformasi:** $T = \begin{bmatrix} 1 & 0 \\ 0 & s \end{bmatrix}$ dengan $s$ turun dari $1 \to 0$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

ModuleNotFoundError: No module named 'numpy'

In [2]:
# Titik-titik dari GeoGebra
points = {
    'A': (2, 3), 'B': (2, 4), 'C': (3, 4), 'D': (3, 3),
    'E': (2, -3), 'F': (3, -3), 'G': (3, -2), 'H': (2, -4),
    'I': (3, -4), 'J': (2, -2), 'K': (2, -1), 'L': (3, -1),
    'M': (2, 2),  'N': (3, 2),  'O': (2, 1),  'P': (3, 1),
}

labels = list(points.keys())
coords = np.array(list(points.values()), dtype=float)
colors = plt.cm.cool(np.linspace(0.2, 0.9, len(labels)))

FRAMES = 120
PAUSE  = 20

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

ax.set_xlim(-1, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='#58a6ff', linewidth=2, alpha=0.4, zorder=2)
ax.axvline(0, color='#30363d', linewidth=1.2, zorder=1)
ax.grid(True, color='#21262d', linewidth=0.5, linestyle='--', alpha=0.6)
ax.tick_params(colors='#8b949e', labelsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#30363d')

ax.set_title('Matrix Transformation → Approaching X-Axis',
             color='#58a6ff', fontsize=13, fontweight='bold', pad=12,
             fontfamily='monospace')
ax.set_xlabel('X', color='#8b949e', fontsize=11)
ax.set_ylabel('Y', color='#8b949e', fontsize=11)

scatters, texts, trails = [], [], []
for i, (label, color) in enumerate(zip(labels, colors)):
    sc, = ax.plot([], [], 'o', color=color, markersize=9,
                  markeredgecolor='white', markeredgewidth=0.8, zorder=5)
    tx  = ax.text(0, 0, label, color=color, fontsize=8, fontweight='bold',
                  ha='left', va='bottom', fontfamily='monospace', zorder=6, alpha=0)
    tr, = ax.plot([], [], '-', color=color, linewidth=1, alpha=0.3, zorder=3)
    scatters.append(sc); texts.append(tx); trails.append(tr)

info_txt = ax.text(0.02, 0.97, '', transform=ax.transAxes,
                   color='#58a6ff', fontsize=10, va='top', ha='left',
                   fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='#161b22',
                             edgecolor='#30363d', alpha=0.9))
mat_txt  = ax.text(0.78, 0.97, '', transform=ax.transAxes,
                   color='#3fb950', fontsize=9, va='top', ha='left',
                   fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='#161b22',
                             edgecolor='#30363d', alpha=0.9))

trail_history = [[] for _ in range(len(labels))]

def get_scale(frame):
    if frame < PAUSE: return 1.0
    if frame >= FRAMES - PAUSE: return 0.0
    t = (frame - PAUSE) / (FRAMES - 2 * PAUSE)
    t = 3*t**2 - 2*t**3  # ease in-out cubic
    return 1.0 - t

def animate(frame):
    s = get_scale(frame)
    for i, (sc, tx, tr) in enumerate(zip(scatters, texts, trails)):
        ox, oy = coords[i]
        ny = oy * s
        sc.set_data([ox], [ny])
        tx.set_position((ox + 0.08, ny + 0.08))
        tx.set_alpha(min(1.0, 1.2 - s * 0.3))
        trail_history[i].append((ox, ny))
        if len(trail_history[i]) > 15:
            trail_history[i].pop(0)
        if len(trail_history[i]) > 1:
            th = np.array(trail_history[i])
            tr.set_data(th[:, 0], th[:, 1])
    info_txt.set_text(f'Compression: {int((1-s)*100):3d}%\nScale Y: {s:.3f}')
    mat_txt.set_text(f'T = [1   0]\n     [0  {s:.2f}]')
    return scatters + texts + trails + [info_txt, mat_txt]

ani = animation.FuncAnimation(
    fig, animate, frames=FRAMES,
    interval=40, blit=True, repeat=True
)

plt.tight_layout()

# Tampilkan inline di notebook & web statis (GitHub Pages)
display(HTML(ani.to_jshtml()))
plt.close()